In [22]:
import os
import pandas as pd
import arcpy
from arcpy.sa import *

from config import PROJECT_FOLDER

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

processed_folder = os.path.join(PROJECT_FOLDER, "data", "processed")
gdb_path = os.path.join(PROJECT_FOLDER, "UrbanHeatEquity.gdb")
output_folder = os.path.join(PROJECT_FOLDER, "outputs")
table_folder = os.path.join(output_folder, "tables")

os.makedirs(table_folder, exist_ok=True)

tracts_study = os.path.join(gdb_path, "phoenix_city_intersecting_tracts")
tracts_landsat = os.path.join(gdb_path, "phoenix_tracts_landsat_crs")

lst_clip_path = os.path.join(processed_folder, "lst_celsius_city_of_phoenix.tif")
canopy_processed = os.path.join(processed_folder, "tree_canopy_city_of_phoenix.tif")
acs_csv = os.path.join(processed_folder, "acs_demographics.csv")

# Zonal statistics: LST

In [23]:
lst_stats_table = os.path.join(gdb_path, "tract_lst_stats")

if arcpy.Exists(lst_stats_table):
    arcpy.management.Delete(lst_stats_table)

ZonalStatisticsAsTable(
    in_zone_data=tracts_landsat,
    zone_field="GEOID",
    in_value_raster=lst_clip_path,
    out_table=lst_stats_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

print("LST zonal statistics complete.")

LST zonal statistics complete.


# Zonal statistics: tree canopy

In [24]:
canopy_stats_table = os.path.join(gdb_path, "tract_canopy_stats")

if arcpy.Exists(canopy_stats_table):
    arcpy.management.Delete(canopy_stats_table)

ZonalStatisticsAsTable(
    in_zone_data=tracts_study,
    zone_field="GEOID",
    in_value_raster=canopy_processed,
    out_table=canopy_stats_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

print("Canopy zonal statistics complete.")

Canopy zonal statistics complete.


# Merge zonal statistics with ACS

In [25]:
lst_df = pd.DataFrame(
    arcpy.da.TableToNumPyArray(lst_stats_table, ["GEOID", "MEAN"])
).rename(columns={"MEAN": "mean_lst_c"})

canopy_df = pd.DataFrame(
    arcpy.da.TableToNumPyArray(canopy_stats_table, ["GEOID", "MEAN"])
).rename(columns={"MEAN": "mean_tree_canopy_pct"})

acs_df = pd.read_csv(acs_csv, dtype={"GEOID": str})

for df in [lst_df, canopy_df, acs_df]:
    df["GEOID"] = (
        df["GEOID"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.zfill(11)
    )
analysis_df = (
    lst_df
    .merge(canopy_df, on="GEOID", how="left")
    .merge(acs_df, on="GEOID", how="left")
)

numeric_fields = [
    "mean_lst_c",
    "mean_tree_canopy_pct",
    "median_income",
    "pct_black",
    "pct_hispanic",
    "pct_poverty",
    "pct_bachelors"
]

for field in numeric_fields:
    analysis_df[field] = pd.to_numeric(analysis_df[field], errors="coerce")

results_csv = os.path.join(table_folder, "tract_heat_equity_results.csv")
analysis_df.to_csv(results_csv, index=False)

print("Merged analysis table saved:", results_csv)
analysis_df.head()

Merged analysis table saved: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\outputs\tables\tract_heat_equity_results.csv


,GEOID,mean_lst_c,mean_tree_canopy_pct,NAME,total_population,median_income,white_alone,black_alone,hispanic_latino,poverty_count,median_home_value,median_gross_rent,bachelors_degree,state,county,tract,pct_black,pct_hispanic,pct_poverty,pct_bachelors
0,04013093104,50.521088,0.256122,Census Tract 931.04; Maricopa County; Arizona,5758,60116.0,1079,1247,3633,1044,61000,1357,195,4,13,93104,21.656825,63.094825,18.131296,3.386593
1,04013103205,48.005625,4.360724,Census Tract 1032.05; Maricopa County; Arizona,2348,128373.0,1853,22,181,149,727900,1935,535,4,13,103205,0.936968,7.708688,6.345826,22.785349
2,04013103206,48.399540,2.617064,Census Tract 1032.06; Maricopa County; Arizona,2201,161932.0,1840,8,219,98,973200,-666666666,613,4,13,103206,0.363471,9.950023,4.452522,27.850977
3,04013104501,49.717108,0.573445,Census Tract 1045.01; Maricopa County; Arizona,3965,50430.0,1197,437,1955,830,276200,1137,167,4,13,104501,11.021438,49.306431,20.933165,4.211854
4,04013104502,51.001902,0.353771,Census Tract 1045.02; Maricopa County; Arizona,5357,56250.0,1691,625,2716,1349,239100,1275,671,4,13,104502,11.666978,50.700019,25.182005,12.525667


# Calculate heat equity priority score

In [26]:
analysis_df["lst_rank"] = analysis_df["mean_lst_c"].rank(pct=True)
analysis_df["low_income_rank"] = (-analysis_df["median_income"]).rank(pct=True)
analysis_df["low_canopy_rank"] = (-analysis_df["mean_tree_canopy_pct"]).rank(pct=True)

analysis_df["heat_equity_priority_score"] = analysis_df[
    ["lst_rank", "low_income_rank", "low_canopy_rank"]
].mean(axis=1, skipna=True)

analysis_df.to_csv(results_csv, index=False)

analysis_df[
    ["GEOID", "mean_lst_c", "mean_tree_canopy_pct", "median_income", "heat_equity_priority_score"]
].head()

,GEOID,mean_lst_c,mean_tree_canopy_pct,median_income,heat_equity_priority_score
0,04013093104,50.521088,0.256122,60116.0,0.801479
1,04013103205,48.005625,4.360724,128373.0,0.129979
2,04013103206,48.399540,2.617064,161932.0,0.137953
3,04013104501,49.717108,0.573445,50430.0,0.708589
4,04013104502,51.001902,0.353771,56250.0,0.830575


# Create final tract feature class and add indicator fields

In [27]:
final_tracts = os.path.join(gdb_path, "heat_equity_tracts_final")

if arcpy.Exists(final_tracts):
    arcpy.management.Delete(final_tracts)

arcpy.management.CopyFeatures(tracts_study, final_tracts)

join_fields = [
    "mean_lst_c",
    "mean_tree_canopy_pct",
    "median_income",
    "pct_black",
    "pct_hispanic",
    "pct_poverty",
    "pct_bachelors",
    "heat_equity_priority_score"
]

existing_fields = [f.name for f in arcpy.ListFields(final_tracts)]

for field in join_fields:
    if field not in existing_fields:
        arcpy.management.AddField(final_tracts, field, "DOUBLE")

analysis_df["GEOID"] = (
    analysis_df["GEOID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(11)
)

lookup = {
    row["GEOID"]: {
        field: None if pd.isna(row[field]) else float(row[field])
        for field in join_fields
    }
    for _, row in analysis_df.iterrows()
}

updated = 0

with arcpy.da.UpdateCursor(final_tracts, ["GEOID"] + join_fields) as cursor:
    for row in cursor:
        geoid = str(row[0]).strip().zfill(11)

        if geoid in lookup:
            for i, field in enumerate(join_fields, start=1):
                row[i] = lookup[geoid][field]
            cursor.updateRow(row)
            updated += 1

print(f"Updated {updated} records.")

Updated 392 records.


# Add readable geodesic area field

In [28]:
if "area_sqkm" not in [f.name for f in arcpy.ListFields(final_tracts)]:
    arcpy.management.AddField(final_tracts, "area_sqkm", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    final_tracts,
    [["area_sqkm", "AREA_GEODESIC"]],
    area_unit="SQUARE_KILOMETERS"
)

print("Area field added.")

Area field added.


# QA checks

In [29]:
total_features = int(arcpy.management.GetCount(final_tracts)[0])
print("Final feature count:", total_features)

qa_fields = join_fields + ["area_sqkm"]

for field in qa_fields:
    null_count = 0

    with arcpy.da.SearchCursor(final_tracts, [field]) as cursor:
        for row in cursor:
            if row[0] is None:
                null_count += 1

    print(f"{field}: {null_count} null values")

Final feature count: 434
mean_lst_c: 42 null values
mean_tree_canopy_pct: 42 null values
median_income: 48 null values
pct_black: 44 null values
pct_hispanic: 44 null values
pct_poverty: 44 null values
pct_bachelors: 44 null values
heat_equity_priority_score: 42 null values
area_sqkm: 0 null values


# Create a cleaned final layer for publishing only

In [30]:
final_clean = os.path.join(gdb_path, "heat_equity_tracts_final_clean")

if arcpy.Exists(final_clean):
    arcpy.management.Delete(final_clean)

where_clause = (
    "mean_lst_c IS NOT NULL AND "
    "mean_tree_canopy_pct IS NOT NULL AND "
    "median_income IS NOT NULL AND "
    "heat_equity_priority_score IS NOT NULL"
)

arcpy.analysis.Select(final_tracts, final_clean, where_clause)

print("Clean publish layer count:", arcpy.management.GetCount(final_clean)[0])

Clean publish layer count: 386
